# 🚗 Proyecto Final de Máster en IA: Aprendizaje por Refuerzo y Conducción Autónoma

Entrenamiento de un agente de **Deep RL** para conducir en el simulador 3D
**Duckietown** usando solo píxeles de la cámara. La nota depende del rendimiento en un
**mapa oculto** con obstáculos (`Duckietown-loop_obstacles-v0`).

### 📋 Fases del proyecto
* **Fase 1 — Calentamiento:** Q-Learning tabular *desde cero* en `FrozenLake-v1` (8x8, resbaladiza).
* **Fase 2 — Baselines:** **DQN** (acción discreta) y **PPO** (acción continua) con Stable-Baselines3.
* **Fase 3 — Algoritmo avanzado:** **SAC** (Soft Actor-Critic, continuo).

### ⚠️ Contrato de evaluación (resumen)
1. Observación **`(1, 64, 64)`** por frame, apilada en **4 frames** → `(4, 64, 64)`.
2. Las clases (`CustomCNN`, wrappers) deben estar definidas e importables al cargar.
3. El profesor carga `best_agent.zip` en un entorno limpio (`requirements.txt`) y
   evalúa en el mapa oculto **`Duckietown-loop_obstacles-v0`** (entrenar ahí = descalificación).

### 🗂️ Organización del código (repositorio)
| Archivo | Contenido |
|---|---|
| `q_learning_sandbox.py` | Fase 1 (Q-Learning tabular) |
| `src/config.py` | Contrato: shapes, mapas, acciones discretas |
| `src/duckie_factory.py` | Entorno base real/mock (capa de abstracción) |
| `src/wrappers.py` | `DuckieWrapper` + `DiscreteActionWrapper` |
| `src/cnn.py` | `CustomCNN` (extractor de características para SB3) |
| `src/envs.py` | `make_env` + `build_vec_env` (FrameStack) |
| `train.py` | Entrenamiento DQN/PPO/SAC |
| `eval.py` | Evaluación de un modelo guardado |
| `scripts/run_training_plan.py` | Lanzador de experimentos |

> Detalle completo del entorno en **`COLAB_SETUP.md`**; experimentos y resultados en
> **`EXPERIMENTS.md`**.

## Dependencias y entorno (Colab limpio)

El kernel de Colab es **Python 3.12**, pero el stack (numpy 1.23.5, gym 0.25.2,
Duckietown…) requiere **Python 3.11**. Por eso montamos un **venv 3.11** y ejecutamos
todo con `{PY}` (subprocess), no con el `pip`/`python` del kernel.

**Ejecutar las celdas de esta sección en orden.**

In [ ]:
# Clonar el repositorio en /content/MAML (repo público).
# Si el repo es privado, usar un PAT con scope 'repo' (no dejar el token en el notebook).
import os
%cd /content
!rm -rf /content/MAML
!git clone https://github.com/AdolfoPM02/MAML.git /content/MAML
%cd /content/MAML
!ls

In [ ]:
# a) Python 3.11 + dependencias de sistema (Duckietown headless)
!sudo add-apt-repository -y ppa:deadsnakes/ppa
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev xvfb python3-opengl ffmpeg freeglut3-dev

In [ ]:
# b) Crear el venv 3.11 y definir PY (intérprete del venv usado en TODO el notebook)
!python3.11 -m venv /content/venv-maml
PY = "/content/venv-maml/bin/python"
!{PY} -m pip install -U pip wheel setuptools
!{PY} --version

In [ ]:
# c) Instalar requirements.txt con el Python 3.11 del venv (NO el kernel 3.12)
%cd /content/MAML
!{PY} -m pip install -r requirements.txt

In [ ]:
# d) Verificación de imports
!MPLBACKEND=Agg {PY} -c "import numpy, torch, gym, gymnasium, stable_baselines3; import gym_duckietown; print('OK', numpy.__version__, torch.__version__)"

## Fase 1 — Q-Learning tabular (`FrozenLake-v1`)
Implementación **desde cero** (sin librerías de Deep RL) de Q-Learning tabular sobre
`FrozenLake-v1` 8x8 resbaladiza: Q-table en numpy, actualización por la **Ecuación de
Bellman**

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma\max_{a'}Q(s',a') - Q(s,a)\,]$$

con política **ε-greedy** y decaimiento, y gráfica de media móvil de recompensa.
Implementación completa en **`q_learning_sandbox.py`** (celda **opcional**, ~30 s):

In [ ]:
!MPLBACKEND=Agg {PY} q_learning_sandbox.py

## Fase 2 — Baselines DQN y PPO (Duckietown)

**Pipeline de visión** (`src/wrappers.py` → `DuckieWrapper`):

`RGB (480×640×3)` → **recorte** del 50% superior (cielo) → **escala de grises** →
**resize 64×64** → `(1, 64, 64)` → **FrameStack(4)** (`src/envs.py`) → `(4, 64, 64)` →
**`CustomCNN`** (`src/cnn.py`).

**Baselines:**
* **DQN** — acción **discreta**: `DiscreteActionWrapper` mapea cada índice a un comando
  `[velocidad, giro]` (`src/config.DISCRETE_ACTIONS`).
* **PPO** — acción **continua**: `Box([-1,-1], [1,1])`.

Ambos usan `CustomCNN` vía `policy_kwargs`. El entrenamiento real se lanza con
`scripts/run_training_plan.py` (ver `EXPERIMENTS.md`); aquí solo los comandos en
**dry-run** (no entrena):

In [ ]:
!{PY} scripts/run_training_plan.py --stage ppo20k --dry-run --eval-after
!{PY} scripts/run_training_plan.py --stage dqn20k --dry-run --eval-after

## Fase 3 — Algoritmo avanzado (SAC)
**SAC** (Soft Actor-Critic) como algoritmo avanzado de acción continua (off-policy,
exploración por entropía). Comparte `CustomCNN` y wrappers con PPO. Dry-run (no entrena):

In [ ]:
!{PY} scripts/run_training_plan.py --stage sac20k --dry-run --eval-after

> **Nota:** aunque SAC se implementó como alternativa avanzada, el **modelo final
> seleccionado fue PPO (20k)** por sus resultados (ver tabla siguiente).

## Resultados finales
Entrenamiento real en Colab (GPU) con el stack validado y `--init-order model-first`.
Evaluación con `eval.py` (3 episodios por mapa):

| modelo | mapa | recompensa (media ± std) | longitud |
|---|---|---|---|
| **ppo20k_gpu — GANADOR** | loop_empty | 960.964 ± 605.380 | 1500.0 |
| **ppo20k_gpu — GANADOR** | small_loop | 317.528 ± 703.254 | 1500.0 |
| **ppo20k_gpu — GANADOR** | loop_obstacles | **1118.216 ± 488.214** | 1500.0 |
| ppo50k_gpu — descartado | loop_empty | −813.731 ± 210.545 | 1500.0 |
| ppo50k_gpu — descartado | small_loop | −1238.875 ± 393.613 | 1500.0 |
| ppo50k_gpu — descartado | loop_obstacles | −1105.378 ± 760.645 | 1500.0 |

**Ganador: `ppo_loop_empty_20k_gpu`** — mejor recompensa en el mapa oculto y positiva en
los tres mapas. `ppo50k_gpu` se descarta (recompensas negativas en los tres mapas).

`Duckietown-loop_obstacles-v0` se usó **solo para evaluación** (`--allow-eval-hidden`),
**nunca para entrenamiento** (bloqueado por `ValueError` en el código).

## Modelo final
El entregable del contrato es **`best_agent.zip`**.

**`best_agent.zip` = copia de `ppo_loop_empty_20k_gpu.zip`** (~4.6 MB), generado en
Colab con:
```bash
cp models/ppo_loop_empty_20k_gpu.zip models/best_agent.zip
```
Se entrega como **artefacto externo** (no versionado en git). *(Durante el desarrollo se
usó el nombre `best_duckie_agent.zip`; el entregable definitivo es `best_agent.zip`.)*

Evaluación final (**opcional**). `best_agent.zip` debe estar en `models/` (o ajustar la
ruta `--model`):

In [ ]:
!env MPLBACKEND=Agg CUDA_VISIBLE_DEVICES="" xvfb-run -a {PY} eval.py --algo ppo --model models/best_agent --map Duckietown-loop_empty-v0 --episodes 1 --device cpu --init-order model-first

## 📦 Entrega
El ZIP final debe contener:

* **`Challenge_RL.ipynb`** — este notebook.
* **`requirements.txt`** — dependencias con versiones fijadas (`==`).
* **`q_learning_sandbox.py`** — Fase 1.
* **`train.py`** — pipeline de entrenamiento (DQN/PPO/SAC).
* **`best_agent.zip`** — modelo final (copia de `ppo_loop_empty_20k_gpu.zip`).
* **`Report.pdf`** — memoria técnica (comparativa de los 3 algoritmos y justificación de la Fase 3).

Opcionalmente, el resto del código: `eval.py`, `src/`, `scripts/`, `COLAB_SETUP.md`,
`EXPERIMENTS.md`.

**Dry-run del contrato recomendado:** Colab limpio → ejecutar la sección *Dependencias*
→ subir `best_agent.zip` a `models/` → ejecutar la celda de *Modelo final*.